In [1]:


import PIL.Image as Image
import os
from io import BytesIO
import base64
import PIL.Image as Image
import pandas as pd
import openai


In [ ]:
dx=pd.read_csv("data.csv")

folders = ['train_val_call', 'train_val_dislike', 'train_val_fist', 'train_val_four', 'train_val_like']

List_of_files = []
files = []
for folder in folders:
    folder_path = f"hagrid-sample-30k-384p/hagrid_30k/{folder}"
    files = sorted(os.listdir(folder_path))
    for file in files:
        folder_file_path = f"{folder_path}/{file}"
        List_of_files.append(folder_file_path)

df = dx.copy()
df["Check"] = df["age"].isnull()
df=df[df["Check"]==True]


In [ ]:
mapping_frame= pd.DataFrame({"file_path": List_of_files})
mapping_frame["id"]= mapping_frame["file_path"].str.split("/").str[-1]
mapping_frame["id"]= mapping_frame["id"].str.split(".").str[0]
df= pd.merge(df, mapping_frame, on="id", how="left")



In [ ]:


#os.environ["OPENAI_API_KEY"]= "Key"
client = openai.Client()


def label_age(encoded):
    response = client.responses.create(
        model="gpt-6-luna",
        reasoning={"effort": "medium"},
        instructions=(
            "Estimate the person's apparent age in years. "
            "Return only a floating-point number, such as 32.1."
        ),
        input=[{
            "role": "user",
            "content": [{
                "type": "input_image",
                "image_url": f"data:image/jpeg;base64,{encoded}",
                "detail": "low",
            }],
        }],
    )
    return float(response.output_text.strip())






In [13]:


missing_data = {}
counter = 0
for i in df["file_path"]:
    counter+=1
    image = Image.open(i).convert("RGB")
    buffer = BytesIO()
    image.convert("RGB").save(buffer, format="JPEG")
    encoded = base64.b64encode(buffer.getvalue()).decode("ascii")
    missing_data[i] = label_age(encoded)
    print(len(df["file_path"])-counter, "images remaining")



179 images remaining
178 images remaining
177 images remaining
176 images remaining
175 images remaining
174 images remaining
173 images remaining
172 images remaining
171 images remaining
170 images remaining
169 images remaining
168 images remaining
167 images remaining
166 images remaining
165 images remaining
164 images remaining
163 images remaining
162 images remaining
161 images remaining
160 images remaining
159 images remaining
158 images remaining
157 images remaining
156 images remaining
155 images remaining
154 images remaining
153 images remaining
152 images remaining
151 images remaining
150 images remaining
149 images remaining
148 images remaining
147 images remaining
146 images remaining
145 images remaining
144 images remaining
143 images remaining
142 images remaining
141 images remaining
140 images remaining
139 images remaining
138 images remaining
137 images remaining
136 images remaining
135 images remaining
134 images remaining
133 images remaining
132 images re

In [14]:
df["age_missing"] = df["file_path"].map(missing_data)
df=df[["id", "age_missing"]]
df.to_csv("missing_age.csv", index=False)